In [ ]:
# configuration class for mouseReMoCo application

from typing import Optional, Tuple
from enum import Enum


class TaskType(Enum):
    """Task types supported by the application"""

    CIRCULAR = "circular"
    LINEAR = "linear"


class Configuration:
    """
    Main configuration class mirroring Java Configuration.java
    Handles all application settings including:
    - Screen and window configuration
    - Circular and linear task parameters
    - Visual styling (colors, cursors, fonts)
    - Input and output settings

    INSTANTIATION: Must be created with NO arguments
        config = Configuration()

    CUSTOMIZATION: Set properties explicitly before setup
        config.cursor_radius = 20
        config.cycle_max_number = 4

    RUNTIME: Call setup.create_and_display() to populate
        - screen_width, screen_height
        - drawable_width, drawable_height
        - center_x, center_y
        - all derived values (internal_limit, external_limit, etc.)
    """

    def __init__(self):
        """Initialize Configuration with NO arguments - all values use defaults.

        To customize behavior:
        1. Create: config = Configuration()
        2. Modify defaults as needed before setup
        3. Call: setup.create_and_display() which populates runtime values

        This design ensures clarity: anything not explicitly set before
        measure_and_correct_dimensions() gets its runtime value there.
        """
        # ===== Window Configuration =====
        self.title = "Wacom Tablet Test"
        self.target_monitor = 2
        self.width = None
        self.height = None
        self.nb_cursor_radii_for_target_margin = 5
        self.software = "mouseReMoCo"

        # ===== Screen & Window Configuration (set during measure_and_correct_dimensions) =====
        self.screen_width = 0
        self.screen_height = 0
        self.drawable_width = 0
        self.drawable_height = 0
        self.frame_location_x = 0
        self.frame_location_y = 0
        self.frame_insets = {"top": 0, "bottom": 0, "left": 0, "right": 0}
        self.frame_undecorated = False
        self.used_screen_id = 0

        # ===== Circular Task Parameters (radii set during measure_and_correct_dimensions) =====
        self.task_string = "circular"
        self.center_x = 0
        self.center_y = 0
        self.corner_x = 0
        self.corner_y = 0
        self.external_radius = 150
        self.internal_radius = 80
        self.border_radius = 1
        self.circle_perimeter_mm = 0

        # ===== Circular task derived values (set during measure_and_correct_dimensions) =====
        self.task_radius = 0.0
        self.tolerance_px = 0
        self.index_of_difficulty = 0.0
        self.internal_limit = 0
        self.external_limit = 0

        # ===== Linear Task Parameters =====
        self.inter_line_distance_mm = 150
        self.line_height_mm = 100
        self.mm2px = 0.0

        # ===== Auditory Rhythm =====
        self.half_period = 2000

        # ===== Cursor Configuration =====
        self.cursor_radius = 16
        self.cursor_color_record = (255, 0, 0)  # RGB red
        r, g, b = self.cursor_color_record
        self.cursor_color_record_outside = (
            max(0, r // 2),
            max(0, g // 2),
            max(0, b // 2),
        )
        self.cursor_color_wait = (255, 255, 0)  # RGB yellow

        # ===== Visual Styling =====
        self.border_color = (255, 255, 255)  # RGB white
        self.background_color = (0, 0, 0)  # RGB black
        self.text_color = (255, 255, 255)  # RGB white

        # ===== Sequence Configuration =====
        self.auto_start = 3600  # seconds before auto start
        self.cycle_max_number = 6  # Move-Rest cycle number
        self.cycle_duration = 20  # seconds for a Move or Rest (half-cycle)
        self.is_target_hidden_during_pause = False

        # ===== Font Configuration =====
        self.font_size = 20
        self.font_family = "Courier"

        # ===== Flags =====
        self.is_with_lsl = False  # Lab Streaming Layer
        self.is_with_pause_target = False

        # ===== Trail Configuration =====
        self.trail_mode = "path_length"  # Active trail mode (1-7)
        self.trail_length = None  # Trail length in pixels; None means 10×cursor_radius
        self.trail_pressure_factor = 2.0  # Multiplier for pressure_scaled mode
        self.trail_speed_factor = 1.0  # Multiplier for speed_adaptive mode
        self.trail_max_age_ms = 200  # Milliseconds for time_based mode
        self.trail_max_point_count = 50  # Max points for point_count mode

        # ===== Application State =====
        self.step = ""

        # Initialize derived values
        self._update_circular_task()

    def _update_circular_task(self):
        """Update circular task derived values"""
        if self.task_string == "circular":
            # Limits of the path
            self.internal_limit = self.internal_radius + self.cursor_radius
            self.external_limit = (
                self.external_radius - self.cursor_radius - self.border_radius
            )

            # ID in the steering law (Accot & Zhai 1999)
            self.task_radius = (self.internal_limit + self.external_limit) / 2.0
            self.tolerance_px = self.external_limit - self.internal_limit

            if self.tolerance_px > 0:
                self.index_of_difficulty = (
                    2.0 * 3.14159 * self.task_radius
                ) / self.tolerance_px

    def get_trail_length(self) -> int:
        """Get trail length for current mode, defaulting to 10×cursor_radius if not set"""
        if self.trail_length is not None:
            return self.trail_length
        return 10 * self.cursor_radius

    def set_index_of_difficulty(self, index_of_difficulty: float):
        """Set index of difficulty and adjust circle parameters"""
        if self.task_string != "circular":
            return

        # Calculate new tolerance width
        w = (3.14159 * self.external_limit) / (index_of_difficulty + 3.14159)
        wn = round(2 * w)

        # Update internal limit and radius
        self.internal_limit = self.external_limit - wn
        self.internal_radius = self.internal_limit - self.cursor_radius

        # Recalculate derived values
        self._update_circular_task()

    def set_circular_task(self):
        """Initialize circular task parameters"""
        self._update_circular_task()

    def set_linear_task(self):
        """Initialize linear task parameters"""
        # Linear task setup would go here
        pass

    def set_circle_perimeter(self, perimeter_mm: int, screen_resolution_ppi: float):
        """Set circle perimeter and adjust circle parameters accordingly"""
        if perimeter_mm <= 0 or screen_resolution_ppi <= 0:
            return

        # Convert mm to pixels
        self.circle_perimeter_mm = perimeter_mm
        perimeter_px = perimeter_mm * screen_resolution_ppi / 25.4  # 25.4 mm per inch

        # Calculate new radius and tolerance
        self.task_radius = perimeter_px / (2.0 * 3.14159)
        tolerance = perimeter_px / self.index_of_difficulty

        external_limit = self.task_radius + tolerance / 2.0
        internal_limit = self.task_radius - tolerance / 2.0

        external_radius = external_limit + self.cursor_radius + self.border_radius
        internal_radius = internal_limit - self.cursor_radius

        self.external_radius = round(external_radius)
        self.internal_radius = round(internal_radius)

        self.corner_x = self.drawable_width // 2 - self.external_radius
        self.corner_y = self.drawable_height // 2 - self.external_radius

        self._update_circular_task()

    def calculate_default_circle_radii(
        self, screen_width: int, screen_height: int
    ) -> tuple[int, int]:
        """Calculate circle radii based on screen dimensions and margin settings"""
        # NOTE: default margin is 5 times cursor radius
        # Calculate available space accounting for margins
        margin_px = self.nb_cursor_radii_for_target_margin * self.cursor_radius
        available_width = screen_width - 2 * margin_px
        available_height = screen_height - 2 * margin_px

        # Use smaller dimension to ensure circle fits
        max_diameter = min(available_width, available_height)

        if max_diameter <= 0:
            return self.external_radius, self.internal_radius

        # External radius is half the maximum diameter
        external_radius = max_diameter // 2

        # Internal radius is 60% of external radius (creates 40% wide tolerance band)
        internal_radius = int(external_radius * 0.6)

        return external_radius, internal_radius

    def set_center_x(self, center_x: int):
        """Set center X and update corner X accordingly"""
        self.center_x = center_x
        self.corner_x = center_x - self.external_radius

    def set_center_y(self, center_y: int):
        """Set center Y and update corner Y accordingly"""
        self.center_y = center_y
        self.corner_y = center_y - self.external_radius

    def set_corner_x(self, corner_x: int):
        """Set corner X and update center X accordingly"""
        self.corner_x = corner_x
        self.center_x = corner_x + self.external_radius

    def set_corner_y(self, corner_y: int):
        """Set corner Y and update center Y accordingly"""
        self.corner_y = corner_y
        self.center_y = corner_y + self.external_radius

    def to_string(self) -> str:
        """Generate configuration string representation"""
        parts = [
            f"software: {self.software}",
            f"title: {self.title}",
            f"targetMonitor: {self.target_monitor}",
            f"isWithLSL: {self.is_with_lsl}",
            f"isWithPauseTarget: {self.is_with_pause_target}",
            f"screenWidth: {self.screen_width}",
            f"screenHeight: {self.screen_height}",
            f"drawableWidth: {self.drawable_width}",
            f"drawableHeight: {self.drawable_height}",
            f"frameLocationX: {self.frame_location_x}",
            f"frameLocationY: {self.frame_location_y}",
            f"frameUndecorated: {self.frame_undecorated}",
            f"usedScreenId: {self.used_screen_id}",
            f"centerX: {self.center_x}",
            f"centerY: {self.center_y}",
            f"marginMultiplier: {self.nb_cursor_radii_for_target_margin}",
            f"task: {self.task_string}",
            f"autoStart: {self.auto_start}",
            f"cycleMaxNumber: {self.cycle_max_number}",
            f"cycleDuration: {self.cycle_duration}",
            f"halfPeriod: {self.half_period}",
            f"borderColor: {self.border_color}",
            f"backgroundColor: {self.background_color}",
            f"textColor: {self.text_color}",
            f"cursorRadius: {self.cursor_radius}",
            f"cursorColorRecord: {self.cursor_color_record}",
            f"cursorColorWait: {self.cursor_color_wait}",
            f"fontSize: {self.font_size}",
            f"fontFamily: {self.font_family}",
            f"trailMode: {self.trail_mode}",
            f"trailLength: {self.trail_length}",
            f"trailPressureFactor: {self.trail_pressure_factor}",
            f"trailSpeedFactor: {self.trail_speed_factor}",
            f"trailMaxAgeMs: {self.trail_max_age_ms}",
            f"trailMaxPointCount: {self.trail_max_point_count}",
        ]

        if self.task_string == "circular":
            parts.extend(
                [
                    f"cornerX: {self.corner_x}",
                    f"cornerY: {self.corner_y}",
                    f"externalRadius: {self.external_radius}",
                    f"internalRadius: {self.internal_radius}",
                    f"internalLimit: {self.internal_limit}",
                    f"externalLimit: {self.external_limit}",
                    f"borderRadius: {self.border_radius}",
                    f"circlePerimeterMm: {self.circle_perimeter_mm}",
                    f"indexOfDifficulty: {self.index_of_difficulty:.2f}",
                    f"taskRadius: {self.task_radius:.2f}",
                    f"taskTolerance: {self.tolerance_px}",
                ]
            )
        elif self.task_string == "linear":
            parts.extend(
                [
                    f"interLineDistanceMm: {self.inter_line_distance_mm}",
                    f"lineHeightMm: {self.line_height_mm}",
                    f"mm2px: {self.mm2px:.2f}",
                ]
            )

        return "\n".join(parts)

In [ ]:
# Screen management — ScreenInfo + ScreenManager utilities
from dataclasses import dataclass

from PyQt6.QtWidgets import QApplication, QWidget


@dataclass
class ScreenInfo:
    """Information about a screen"""

    name: str
    index: int
    width: int
    height: int
    pos_x: int
    pos_y: int
    phys_width_mm: float
    phys_height_mm: float
    dpi_x: float
    dpi_y: float
    dpi_avg: float
    diag_inches: float


class ScreenManager:
    """Static utility methods for screen and window management"""

    @staticmethod
    def get_screen_info(screen, app: QApplication) -> ScreenInfo:
        """Extract detailed info from a QScreen object"""
        geometry = screen.geometry()
        phys_size = screen.physicalSize()

        # Calculate DPI
        dpi_x = geometry.width() / (phys_size.width() / 25.4)
        dpi_y = geometry.height() / (phys_size.height() / 25.4)
        dpi_avg = (dpi_x + dpi_y) / 2

        # Calculate diagonal in inches
        diag_inches = (phys_size.width() ** 2 + phys_size.height() ** 2) ** 0.5 / 25.4

        return ScreenInfo(
            name=screen.name(),
            index=app.screens().index(screen),
            width=geometry.width(),
            height=geometry.height(),
            pos_x=geometry.x(),
            pos_y=geometry.y(),
            phys_width_mm=phys_size.width(),
            phys_height_mm=phys_size.height(),
            dpi_x=dpi_x,
            dpi_y=dpi_y,
            dpi_avg=dpi_avg,
            diag_inches=diag_inches,
        )

    @staticmethod
    def get_all_screens(app: QApplication) -> list[ScreenInfo]:
        """Get info for all connected screens"""
        return [ScreenManager.get_screen_info(screen, app) for screen in app.screens()]

    @staticmethod
    def print_all_screens(screens: list[ScreenInfo]):
        """Print formatted screen information"""
        print("=" * 60)
        for s in screens:
            print(f"\nScreen {s.index + 1}: {s.name}")
            print(f"  Geometry: {s.width}×{s.height} @ ({s.pos_x}, {s.pos_y})")
            print(f"  DPI: {s.dpi_x:.1f}×{s.dpi_y:.1f} (avg: {s.dpi_avg:.1f})")
            print(f"  Physical: {s.phys_width_mm:.1f}×{s.phys_height_mm:.1f} mm")
            print(f'  Diagonal: {s.diag_inches:.1f}"')

    @staticmethod
    def get_target_screen(
        screens: list[ScreenInfo], config: Configuration
    ) -> ScreenInfo:
        """Get the target screen with safe fallback"""
        target_index = config.target_monitor - 1  # Convert 1-indexed to 0-indexed
        if 0 <= target_index < len(screens):
            return screens[target_index]
        print(f"⚠ Monitor {config.target_monitor} not found, using primary screen")
        return screens[0]

    @staticmethod
    def get_usable_screen_size(
        app: QApplication, screen_info: ScreenInfo
    ) -> tuple[int, int]:
        """Get usable screen size (excludes taskbars, etc.)"""
        screen = app.screens()[screen_info.index]
        usable = screen.availableGeometry()
        return usable.width(), usable.height()

    @staticmethod
    def get_window_drawable_area(
        widget: QWidget, initial_width: int, initial_height: int
    ) -> tuple[int, int, dict]:
        """Calculate actual drawable area accounting for window frame insets"""
        frame_geometry = widget.frameGeometry()
        content_geometry = widget.geometry()

        # Calculate frame insets
        insets = {
            "top": content_geometry.top() - frame_geometry.top(),
            "bottom": frame_geometry.bottom() - content_geometry.bottom(),
            "left": content_geometry.left() - frame_geometry.left(),
            "right": frame_geometry.right() - content_geometry.right(),
        }

        # Calculate actual drawable dimensions
        actual_width = initial_width - insets["left"] - insets["right"]
        actual_height = initial_height - insets["top"] - insets["bottom"]

        return actual_width, actual_height, insets


class TabletDetector:
    """Detect graphics tablets using Qt's QInputDevice"""

    @staticmethod
    def get_tablets():
        """Returns list of detected stylus/tablet devices"""
        from PyQt6.QtGui import QInputDevice

        tablets = []
        for device in QInputDevice.devices():
            if device.type() == QInputDevice.DeviceType.Stylus:
                tablets.append(device.name())
        return tablets

    @staticmethod
    def has_tablet():
        """Returns True if any tablet is detected"""
        return len(TabletDetector.get_tablets()) > 0

    @staticmethod
    def print_tablet_status():
        """Print tablet detection status to console"""
        tablets = TabletDetector.get_tablets()
        print("\n" + "="*60)
        if tablets:
            print(f"✓ Tablet detected: {tablets[0]}")
            if len(tablets) > 1:
                print(f"  ({len(tablets)} total devices found)")
        else:
            print("⚠ No tablet detected - will use mouse input only")
        print("="*60)

    @staticmethod
    def print_all_devices():
        """Debug: Print all input devices Qt sees"""
        from PyQt6.QtGui import QInputDevice
        print("\n" + "="*60)
        print("All detected input devices:")
        devices = QInputDevice.devices()
        if not devices:
            print("  (no devices found)")
        else:
            for device in devices:
                print(f"  - {device.name()}: {device.type()}")
        print("="*60)

In [ ]:
# Cursor Factory — Generate custom cursor images

from PyQt6.QtGui import QPixmap, QPainter, QColor, QCursor
from PyQt6.QtCore import Qt, QPoint


class CursorFactory:
    """Factory for creating custom cursor images with filled circles and crosshairs"""

    @staticmethod
    def create_cursor(
        radius: int,
        color: tuple[int, int, int],
        background_color: tuple[int, int, int] = (0, 0, 0),
    ) -> QCursor:
        """
        Create a custom cursor with a filled circle and center crosshair.

        Args:
            radius: Cursor circle radius in pixels
            color: RGB tuple (r, g, b) for circle color
            background_color: RGB tuple for background (for crosshair visibility)

        Returns:
            QCursor with the custom cursor image
        """
        diameter = radius * 2

        # Create transparent pixmap
        pixmap = QPixmap(diameter, diameter)
        pixmap.fill(Qt.GlobalColor.transparent)

        # Create painter and draw on pixmap
        painter = QPainter(pixmap)
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw filled circle
        circle_color = QColor(*color)
        painter.setBrush(circle_color)
        painter.setPen(circle_color)
        painter.drawEllipse(0, 0, diameter, diameter)

        # Draw center crosshair (two perpendicular lines)
        crosshair_color = QColor(*background_color)
        painter.setPen(crosshair_color)

        crosshair_length = 4  # pixels extending from center in each direction
        center = radius

        # Horizontal line
        painter.drawLine(
            center - crosshair_length, center, center + crosshair_length, center
        )

        # Vertical line
        painter.drawLine(
            center, center - crosshair_length, center, center + crosshair_length
        )

        painter.end()

        # Create cursor with hotspot at center
        hotspot = QPoint(radius, radius)
        cursor = QCursor(pixmap, hotspot.x(), hotspot.y())

        return cursor

    @staticmethod
    def create_record_cursor(config: "Configuration") -> QCursor:
        """Create cursor for recording state (red circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_record,
            background_color=config.background_color,
        )

    @staticmethod
    def create_wait_cursor(config: "Configuration") -> QCursor:
        """Create cursor for waiting state (yellow circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_wait,
            background_color=config.background_color,
        )

    @staticmethod
    def create_out_cursor(config: "Configuration") -> QCursor:
        """Create cursor for outside target state (darkened record color)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_record_outside,
            background_color=config.background_color,
        )

In [ ]:
# Circular target rendering

from dataclasses import dataclass

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QBrush, QColor, QPainter, QPen


@dataclass
class CircularTaskConfig:
    """Configuration for circular target task"""

    external_radius: int = 150  # pixels
    internal_radius: int = 80  # pixels
    background_color: str = "black"
    path_color: str = "#333333"  #  darkgray < "#333333"  < "#1a1a1a" < black
    circle_border_color: str = "white"
    circle_border_width: int = 2

    @staticmethod
    def rgb_to_hex(rgb_tuple: tuple[int, int, int]) -> str:
        """Convert RGB tuple (r, g, b) to hex color string"""
        r, g, b = rgb_tuple
        return f"#{r:02x}{g:02x}{b:02x}"


class CircularTargetWidget:
    """Draw circular target with tolerance band"""

    def __init__(self, config: CircularTaskConfig = None):
        self.config = config or CircularTaskConfig()

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the circular target"""
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw external circle (border)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.external_radius,
            fill_color=self.config.path_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

        # Draw internal circle (background)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.internal_radius,
            fill_color=self.config.background_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

    def _draw_filled_circle(
        self,
        painter: QPainter,
        x: int,
        y: int,
        radius: int,
        fill_color: str,
        border_color: str,
        border_width: int,
    ):
        """Helper to draw filled circle with border"""
        # Set fill color
        fill = QColor(fill_color)
        painter.setBrush(QBrush(fill))

        # Set border (pen)
        border = QColor(border_color)
        pen = QPen(border)
        pen.setWidth(border_width)
        painter.setPen(pen)

        # Draw circle
        painter.drawEllipse(x - radius, y - radius, 2 * radius, 2 * radius)

In [ ]:
# Window setup orchestration


class WindowSetup:
    """Encapsulates the complete window setup and initialization process.

    Manages both window AND configuration lifecycle, making WindowSetup
    the true orchestrator of the complete initialization workflow.

    Configuration is created internally and can be customized via the
    config property before calling create_and_display().
    """

    def __init__(
        self,
        app: QApplication,
        tablet_test_class: type,
    ):
        self.app = app
        self.tablet_test_class = tablet_test_class

        # Create configuration internally - owned by WindowSetup
        self.config = Configuration()

        self.screens = None
        self.target_screen_info = None
        self.usable_width = None
        self.usable_height = None
        self.widget = None

    def initialize_screens(self):
        """Step 1: Detect screens and select target"""
        self.screens = ScreenManager.get_all_screens(self.app)
        ScreenManager.print_all_screens(self.screens)
        self.target_screen_info = ScreenManager.get_target_screen(
            self.screens, self.config
        )
        self.usable_width, self.usable_height = ScreenManager.get_usable_screen_size(
            self.app, self.target_screen_info
        )

    def calculate_initial_radii(self) -> tuple[int, int, "CircularTaskConfig"]:
        """Step 2-3: Calculate initial radii and create circle config"""
        external_radius, internal_radius = self.config.calculate_default_circle_radii(
            screen_width=self.usable_width,
            screen_height=self.usable_height,
        )

        circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )
        return external_radius, internal_radius, circle_config

    def create_widget(self) -> QWidget:
        """Step 4: Create and position window widget"""
        widget = self.tablet_test_class(
            self.target_screen_info,
            self.circle_config,
            self.usable_width,
            self.usable_height,
            config=self.config,
            window_setup=self,
            output_data=None, 
        )
        widget.setWindowTitle(self.config.title)
        widget.move(self.target_screen_info.pos_x, self.target_screen_info.pos_y)

        window_width = self.config.width or self.usable_width
        window_height = self.config.height or self.usable_height
        widget.resize(window_width, window_height)

        return widget

    def measure_and_correct_dimensions(self):
        """Step 5-8: Measure frame insets and update widget with corrected dimensions"""
        # Show window to make frame insets calculable
        self.widget.show()
        self.app.processEvents()

        # Measure actual drawable area
        actual_width, actual_height, insets = ScreenManager.get_window_drawable_area(
            self.widget, self.usable_width, self.usable_height
        )
        print(
            f"\nWindow frame insets: Top={insets['top']}, Bottom={insets['bottom']}, Left={insets['left']}, Right={insets['right']}"
        )

        # Recalculate radii with actual drawable area
        external_radius, internal_radius = self.config.calculate_default_circle_radii(
            actual_width, actual_height
        )

        # Update config with corrected radii and actual dimensions
        self.config.screen_width = self.target_screen_info.width
        self.config.screen_height = self.target_screen_info.height
        self.config.drawable_width = actual_width
        self.config.drawable_height = actual_height
        self.config.frame_location_x = self.target_screen_info.pos_x
        self.config.frame_location_y = self.target_screen_info.pos_y
        self.config.frame_insets = insets
        self.config.used_screen_id = self.target_screen_info.index
        self.config.external_radius = external_radius
        self.config.internal_radius = internal_radius

        # Calculate and set center coordinates
        center_x = actual_width // 2
        center_y = actual_height // 2
        self.config.set_center_x(center_x)
        self.config.set_center_y(center_y)

        # Update derived values
        self.config._update_circular_task()

        # Create corrected circle config
        corrected_circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )

        # Update widget with corrected values
        self.widget.circular_target = CircularTargetWidget(
            config=corrected_circle_config
        )
        self.widget.drawable_width = actual_width
        self.widget.drawable_height = actual_height
        self.widget.center_x = center_x
        self.widget.center_y = center_y
        self.widget.update()

    def finalize_display(self):
        """Step 9: Finalize window display and print configuration"""
        # Create OutputData NOW = with the corrected configuration 
        self.output_data = OutputTablet(
            config=self.config,
            enable_csv=True,      # Enable CSV output
            enable_lsl=False      # Set to True when LSL library available
        )

        # Assign output_data to widget and its components
        self.widget.output_data = self.output_data
        self.widget.trail_mouse.output_data = self.output_data
        self.widget.trail_tablet.output_data = self.output_data

        # Bring window to front and focus
        self.widget.raise_()
        self.widget.setFocus()

        # Display the final configuration
        print("\n" + "=" * 60)
        print(self.config.to_string())
        print("=" * 60 + "\n")

    def update_configuration(self, **kwargs):
        """Update configuration parameters and refresh the display.

        Args:
            **kwargs: Configuration parameters to update
                e.g., update_configuration(cursor_radius=20, index_of_difficulty=100)

        Supports any configuration property:
            - cursor_radius, cycle_max_number, background_color, etc.
            - index_of_difficulty (calls set_index_of_difficulty internally)
            - circle_perimeter (tuple: (perimeter_mm, screen_resolution_ppi))
        """
        for key, value in kwargs.items():
            if key == "index_of_difficulty":
                # Special handling for index_of_difficulty
                self.config.set_index_of_difficulty(value)
            elif key == "circle_perimeter":
                # Expects tuple: (perimeter_mm, screen_resolution_ppi)
                perimeter_mm, screen_resolution_ppi = value
                self.config.set_circle_perimeter(perimeter_mm, screen_resolution_ppi)
            elif hasattr(self.config, key):
                setattr(self.config, key, value)
            else:
                print(f"⚠ Warning: Configuration has no attribute '{key}'")

        # Update circular task derived values
        self.config._update_circular_task()

        # Recreate circle config with updated radii if needed
        corrected_circle_config = CircularTaskConfig(
            external_radius=self.config.external_radius,
            internal_radius=self.config.internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )

        # Update widget
        self.widget.circular_target = CircularTargetWidget(
            config=corrected_circle_config
        )
        self.widget.update()

    def create_and_display(self) -> QWidget:
        """Execute the complete setup pipeline"""
        self.initialize_screens()
        _, _, self.circle_config = self.calculate_initial_radii()
        self.widget = self.create_widget()
        self.measure_and_correct_dimensions()
        self.finalize_display()
        return self.widget

In [ ]:
# Trail Class — Comet trail rendering and management

from collections import deque
import time

from PyQt6.QtGui import QColor, QPainter, QPen


class Trail:
    """
    Manages comet trail rendering with 7 different computation modes.

    Modes:
    1. fixed - constant length trail
    2. pressure_scaled - trail length scales with tablet pressure
    3. speed_adaptive - trail length scales with cursor velocity
    4. time_based - points fade based on age (milliseconds)
    5. point_count - keep last N points
    6. combined - pressure and speed multipliers combined
    7. path_length - fade based on cumulative distance traveled

    Trail data structure: deque of (x, y, timestamp, cumulative_distance, pressure)
    Each point remembers the pressure when it was recorded.
    """

    # Single source of truth for all valid trail modes
    VALID_MODES = [
        "fixed",
        "pressure_scaled",
        "speed_adaptive",
        "time_based",
        "point_count",
        "combined",
        "path_length",
    ]

    def __init__(self, config: "Configuration", is_mouse: bool = False):
        """Initialize Trail with configuration reference and input type flag"""
        self.config = config
        self.is_mouse = is_mouse  # Track if this is mouse or tablet trail
        self.trail = deque()  # (x, y, timestamp, cumulative_path_distance, pressure)
        self.current_pressure = 0.0
        self.cumulative_distance = 0.0
        self._current_speed = 0.0
        self._last_x = None
        self._last_y = None
        self.output_data = None  # Set by MainWindow after OutputData is created

    def add_point(self, x: float, y: float):  # x, y now float
        """Add point with subpixel coordinates"""
        if self._last_x is not None and self._last_y is not None:
            dx = x - self._last_x
            dy = y - self._last_y
            distance = (dx**2 + dy**2) ** 0.5
            self._current_speed = distance
            self.cumulative_distance += distance
        else:
            self._current_speed = 0.0

        # Add point with current timestamp, cumulative distance, and pressure
        self.trail.append(
            (x, y, time.time(), self.cumulative_distance, self.current_pressure)
        )

        # Update last position
        self._last_x = x
        self._last_y = y

        # Write data to CSV if OutputData is available
        if self.output_data:
            timestamp_ms = int(time.time() * 1000)
            # Calculate if point is inside target circle
            dx = self.config.center_x - x
            dy = self.config.center_y - y
            distance_to_center = (dx * dx + dy * dy) ** 0.5
            is_inside = self.config.internal_limit < distance_to_center < self.config.external_limit
            
            self.output_data.write_data(
                timestamp_ms=timestamp_ms, x=x, y=y, is_inside=is_inside,
                pressure=self.current_pressure, tilt_x=0.0, tilt_y=0.0
            )

        # Prune based on current mode
        self.prune()

    def prune(self):
        """Remove old points from trail based on current mode"""
        mode = self.config.trail_mode

        if mode in ["fixed", "pressure_scaled", "speed_adaptive", "combined"]:
            # Distance-based pruning (Euclidean)
            if len(self.trail) > 0:
                current_x, current_y = self.trail[-1][0], self.trail[-1][1]
                trail_length = self.get_length()
                self.trail = deque(
                    (x, y, t, d, p)
                    for x, y, t, d, p in self.trail
                    if (x - current_x) ** 2 + (y - current_y) ** 2 <= trail_length**2
                )

        elif mode == "time_based":
            # Time-based pruning
            now = time.time()
            cutoff_time = now - (self.config.trail_max_age_ms / 1000.0)
            self.trail = deque(
                (x, y, t, d, p) for x, y, t, d, p in self.trail if t >= cutoff_time
            )

        elif mode == "point_count":
            # Keep only last N points
            while len(self.trail) > self.config.trail_max_point_count:
                self.trail.popleft()

        elif mode == "path_length":
            # Path-distance-based pruning
            threshold = self.get_length()
            self.trail = deque(
                (x, y, t, d, p)
                for x, y, t, d, p in self.trail
                if self.cumulative_distance - d <= threshold
            )

    def get_length(self) -> float:
        """Compute trail length based on current mode"""
        base = self.config.get_trail_length()  # 10×cursor_radius or explicit value

        mode = self.config.trail_mode

        if mode == "fixed":
            return base

        elif mode == "pressure_scaled":
            return base * (
                1 + self.config.trail_pressure_factor * self.current_pressure
            )

        elif mode == "speed_adaptive":
            return base * (
                1 + self.config.trail_speed_factor * self._current_speed / 500
            )

        elif mode == "time_based":
            return float("inf")  # pruned by time, not distance

        elif mode == "point_count":
            return float("inf")  # pruned by count, not distance

        elif mode == "combined":
            pressure_mult = (
                1 + self.config.trail_pressure_factor * self.current_pressure
            )
            speed_mult = 1 + self.config.trail_speed_factor * self._current_speed / 500
            return base * pressure_mult * speed_mult

        elif mode == "path_length":
            return base  # Uses cumulative distance, not Euclidean

        return base

    def _get_color_for_position(self, x: int, y: int) -> tuple[int, int, int]:
        """Determine trail color based on position relative to target"""
        dx = self.config.center_x - x
        dy = self.config.center_y - y
        distance = (dx * dx + dy * dy) ** 0.5

        is_inside = self.config.internal_limit < distance < self.config.external_limit

        # Mouse trails use blue, tablet trails use red
        if self.is_mouse:
            # Blue colors for mouse trail
            if is_inside:
                return (0, 0, 255)  # Bright blue inside
            else:
                return (0, 0, 128)  # Dark blue outside
        else:
            # Red colors for tablet trail
            if is_inside:
                return self.config.cursor_color_record  # Red inside
            else:
                return self.config.cursor_color_record_outside  # Dark red outside

    def _draw_segment(
        self,
        painter: QPainter,
        x1: int,
        y1: int,
        x2: int,
        y2: int,
        pressure: float,
        opacity: float,
    ) -> bool:
        """Draw a single trail segment with thickness check and position-based coloring.

        Returns True if segment was drawn, False if skipped (sub-pixel thickness).
        Mouse vs tablet trails use different thickness calculations:
        - Mouse: Fixed 5px width
        - Tablet: 2 × cursor_radius × pressure (pressure-dependent)
        """
        # Calculate thickness based on trail type
        if self.is_mouse:
            thickness = 5  # Fixed width for mouse trails
        else:
            thickness = 2 * self.config.cursor_radius * pressure  # Pressure-scaled for tablet
        
        if thickness < 1:
            return False

        # Get position-based color and set alpha
        color = QColor(*self._get_color_for_position(x2, y2))
        color.setAlpha(int(255 * opacity))

        # Draw the segment
        painter.setPen(QPen(color, thickness))
        painter.drawLine(int(x1), int(y1), int(x2), int(y2))
        return True

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the entire trail with mode-specific opacity"""
        if len(self.trail) < 2:
            return

        trail_list = list(self.trail)
        current_x, current_y = trail_list[-1][0], trail_list[-1][1]
        mode = self.config.trail_mode

        # Render based on mode
        if mode in ["fixed", "pressure_scaled", "speed_adaptive", "combined"]:
            self._draw_distance_based(painter, trail_list, current_x, current_y)

        elif mode == "time_based":
            self._draw_time_based(painter, trail_list)

        elif mode == "point_count":
            self._draw_count_based(painter, trail_list)

        elif mode == "path_length":
            self._draw_path_distance_based(painter, trail_list)

    def _draw_distance_based(
        self, painter: QPainter, trail_list: list, current_x: int, current_y: int
    ):
        """Draw trail with distance-based opacity fade"""
        trail_length = self.get_length()
        for i in range(len(trail_list) - 1):
            x1, y1, _, _, pressure1 = trail_list[i]
            x2, y2, _, _, pressure2 = trail_list[i + 1]

            distance = ((x2 - current_x) ** 2 + (y2 - current_y) ** 2) ** 0.5
            opacity = max(0, 1 - distance / trail_length)

            self._draw_segment(painter, x1, y1, x2, y2, pressure2, opacity)

    def _draw_time_based(self, painter: QPainter, trail_list: list):
        """Draw trail with time-based opacity fade"""
        now = time.time()
        for i in range(len(trail_list) - 1):
            x1, y1, t1, _, pressure1 = trail_list[i]
            x2, y2, t2, _, pressure2 = trail_list[i + 1]

            age = (now - t2) * 1000  # milliseconds
            opacity = max(0, 1 - age / self.config.trail_max_age_ms)

            self._draw_segment(painter, x1, y1, x2, y2, pressure2, opacity)

    def _draw_count_based(self, painter: QPainter, trail_list: list):
        """Draw trail with count-based opacity fade"""
        total_points = len(trail_list)
        for i in range(len(trail_list) - 1):
            x1, y1, _, _, pressure1 = trail_list[i]
            x2, y2, _, _, pressure2 = trail_list[i + 1]

            opacity = (i + 1) / total_points

            self._draw_segment(painter, x1, y1, x2, y2, pressure2, opacity)

    def _draw_path_distance_based(self, painter: QPainter, trail_list: list):
        """Draw trail with path-distance-based opacity fade"""
        threshold = self.get_length()
        for i in range(len(trail_list) - 1):
            x1, y1, _, d1, pressure1 = trail_list[i]
            x2, y2, _, d2, pressure2 = trail_list[i + 1]

            # Path distance from newest point
            path_distance = self.cumulative_distance - d2
            opacity = max(0, 1 - path_distance / threshold)

            self._draw_segment(painter, x1, y1, x2, y2, pressure2, opacity)

    def clear(self):
        """Clear all trail points and reset distance tracking"""
        self.trail.clear()
        self.cumulative_distance = 0.0
        self._current_speed = 0.0
        self._last_x = None
        self._last_y = None

    def set_mode(self, mode_name: str):
        """Switch to a new trail mode and clear trail"""
        if mode_name in self.VALID_MODES:
            self.config.trail_mode = mode_name
            self.clear()

In [ ]:
# OutputData class — Manages CSV data output for mouse/tablet tracking

import csv
from datetime import datetime
from pathlib import Path
import time


class OutputData:
    """
    Manages CSV data output for mouse/tablet tracking.
    Mirrors Java OutputMouse.java functionality.
    Creates two CSV files:
    - data.csv: Mouse/tablet position events with timestamp, x, y, isInTarget, pressure, tiltX, tiltY
    - marker.csv: Event markers and phase transitions with timestamp, milliseconds, marker
    """

    def __init__(self, config, data_filename='data.csv', marker_filename='marker.csv'):
        """
        Initialize output files with configuration header.
        
        Args:
            config: Configuration object containing screen and task parameters
            data_filename: Name of the data CSV file
            marker_filename: Name of the marker CSV file
        """
        self.config = config
        self.data_filename = data_filename
        self.marker_filename = marker_filename
        self.creation_timestamp = datetime.now()

        # Initialize CSV file handles and writers
        self.data_file = None
        self.marker_file = None
        self.data_writer = None
        self.marker_writer = None

        # Create and initialize files
        self._init_files()

    def _init_files(self):
        """Create and write headers to both CSV files"""
        # Create data.csv
        self.data_file = open(self.data_filename, 'w', newline='')
        self.data_writer = csv.writer(self.data_file)

        # Write configuration header
        config_line = self._config_to_string()
        self.data_file.write(config_line + '\n')

        # Write creation timestamp
        timestamp_str = self.creation_timestamp.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] + ' +0100'
        self.data_file.write(timestamp_str + '\n')
        self.data_file.write('\n')

        # Write column headers (includes pressure and tilt data)
        self.data_writer.writerow(['timestamp', 'mouseX', 'mouseY', 'mouseInTarget', 'pressure', 'tiltX', 'tiltY'])
        self.data_file.flush()

        # Create marker.csv
        self.marker_file = open(self.marker_filename, 'w', newline='')
        self.marker_writer = csv.writer(self.marker_file)

        # Write same configuration and timestamp headers to marker file
        self.marker_file.write(config_line + '\n')
        self.marker_file.write(timestamp_str + '\n')
        self.marker_file.write('\n')

        # Write marker column headers
        self.marker_writer.writerow(['timestamp', 'milliseconds', 'marker'])
        self.marker_file.flush()

        print(f"✓ Created {self.data_filename} and {self.marker_filename}")

    def _config_to_string(self) -> str:
        """
        Convert configuration to semicolon-separated string for CSV header.
        Mirrors Java Configuration.toString() format.
        
        Returns:
            Semicolon-separated configuration string
        """
        config_dict = {
            'software': 'mouseReMoCo',
            'version': '2.0.0-python',
            'screenWidth': self.config.screen_width,
            'screenHeight': self.config.screen_height,
            'centerX': self.config.center_x,
            'centerY': self.config.center_y,
            'isWithLSL': str(self.config.is_with_lsl).lower(),
        }

        # Add task-specific configuration
        if self.config.task_string == 'circular':
            config_dict.update({
                'externalRadius': self.config.external_radius,
                'internalRadius': self.config.internal_radius,
                'internalLimit': self.config.internal_limit,
                'externalLimit': self.config.external_limit,
                'borderRadius': self.config.border_radius,
                'circlePerimeterMm': self.config.circle_perimeter_mm,
                'indexOfDifficulty': round(self.config.index_of_difficulty, 2),
                'taskRadius': round(self.config.task_radius, 2),
                'taskTolerance': self.config.tolerance_px,
            })
        elif self.config.task_string == 'linear':
            config_dict.update({
                'interLineDistanceMm': self.config.inter_line_distance_mm,
                'lineHeightMm': self.config.line_height_mm,
                'mm2px': round(self.config.mm2px, 2),
            })

        # Format as semicolon-separated key-value pairs
        return ';'.join([f"{k} {v}" for k, v in config_dict.items()])

    def write_data(self, timestamp_ms: int, x: int, y: int, is_inside: bool, 
                   pressure: float = 0.0, tilt_x: float = 0.0, tilt_y: float = 0.0):
        """
        Write position data point to data.csv with pressure and tilt information.
        
        Args:
            timestamp_ms: Timestamp in milliseconds (from event or system time)
            x: Mouse/tablet X coordinate (pixels)
            y: Mouse/tablet Y coordinate (pixels)
            is_inside: Boolean flag indicating if position is inside target
            pressure: Tablet pressure value (0.0 to 1.0, 0 for mouse input)
            tilt_x: Tablet tilt in X direction (-90 to +90 degrees, 0 for mouse)
            tilt_y: Tablet tilt in Y direction (-90 to +90 degrees, 0 for mouse)
        """
        try:
            self.data_writer.writerow([
                timestamp_ms,
                x,
                y,
                1 if is_inside else 0,
                round(pressure, 3),
                round(tilt_x, 2),
                round(tilt_y, 2)
            ])
            self.data_file.flush()
        except Exception as e:
            print(f"ERROR writing data to {self.data_filename}: {e}")

    def write_marker(self, marker_text: str):
        """
        Write event marker to marker.csv with automatic timestamp.
        
        Event examples:
        - 'TabletDetected': When tablet is first detected
        - 'ModeChanged_fixed': When trail mode changes
        - 'TaskStart': When task begins
        - 'CircleEntered': When cursor enters target circle
        - 'CircleExited': When cursor leaves target circle
        
        Args:
            marker_text: Event description string
        """
        try:
            current_time = datetime.now()
            timestamp_ms = int(current_time.timestamp() * 1000)
            timestamp_str = current_time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            
            self.marker_writer.writerow([
                timestamp_str,
                timestamp_ms,
                marker_text
            ])
            self.marker_file.flush()
        except Exception as e:
            print(f"ERROR writing marker to {self.marker_filename}: {e}")

    def close(self):
        """
        Close both CSV files gracefully.
        
        Call this method when application terminates or task completes
        to ensure all data is written and file handles are released.
        Should be called in MainWindow.closeEvent() or application cleanup.
        """
        try:
            if self.data_file:
                self.data_file.close()
                print(f"✓ Closed {self.data_filename}")
            if self.marker_file:
                self.marker_file.close()
                print(f"✓ Closed {self.marker_filename}")
        except Exception as e:
            print(f"ERROR closing files: {e}")

    def __del__(self):
        """
        Destructor - ensures files are closed when object is garbage collected.
        
        This is a safety net in case close() is not explicitly called.
        Python's garbage collector will invoke this when OutputData goes out of scope.
        """
        self.close()


In [ ]:
# OutputTablet class -- Manage output of tablet/mouse to CSV files or to LSL stream

# Output Backend Architecture — With subpixel coordinate support

from abc import ABC, abstractmethod
import csv
from datetime import datetime
from pathlib import Path


class OutputBackend(ABC):
    """Abstract base class for output targets (CSV, LSL, etc.)"""
    
    @abstractmethod
    def write_data(self, timestamp_ms: int, x: float, y: float, is_inside: bool, 
                   pressure: float = 0.0, tilt_x: float = 0.0, tilt_y: float = 0.0):
        """Write position data point with subpixel precision"""
        pass
    
    @abstractmethod
    def write_marker(self, marker_text: str):
        """Write event marker"""
        pass
    
    @abstractmethod
    def close(self):
        """Close backend resources"""
        pass


class CSVBackend(OutputBackend):
    """CSV file output backend (data.csv, marker.csv) with subpixel support"""
    
    def __init__(self, config, data_filename='data.csv', marker_filename='marker.csv'):
        self.config = config
        self.data_filename = data_filename
        self.marker_filename = marker_filename
        self.creation_timestamp = datetime.now()
        
        self.data_file = None
        self.marker_file = None
        self.data_writer = None
        self.marker_writer = None
        
        self._init_files()
    
    def _init_files(self):
        """Create and write headers to both CSV files"""
        # Create data.csv
        self.data_file = open(self.data_filename, 'w', newline='')
        self.data_writer = csv.writer(self.data_file)
        
        # Write configuration header
        config_line = self._config_to_string()
        self.data_file.write(config_line + '\n')
        
        # Write creation timestamp
        timestamp_str = self.creation_timestamp.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] + ' +0100'
        self.data_file.write(timestamp_str + '\n')
        self.data_file.write('\n')
        
        # Write column headers (updated to reflect subpixel precision)
        self.data_writer.writerow(['timestamp', 'mouseX', 'mouseY', 'mouseInTarget', 'pressure', 'tiltX', 'tiltY'])
        self.data_file.flush()
        
        # Create marker.csv
        self.marker_file = open(self.marker_filename, 'w', newline='')
        self.marker_writer = csv.writer(self.marker_file)
        
        # Write same headers to marker file
        self.marker_file.write(config_line + '\n')
        self.marker_file.write(timestamp_str + '\n')
        self.marker_file.write('\n')
        
        # Write marker column headers
        self.marker_writer.writerow(['timestamp', 'milliseconds', 'marker'])
        self.marker_file.flush()
        
        print(f"✓ CSV Backend: Created {self.data_filename} and {self.marker_filename}")
    
    def _config_to_string(self) -> str:
        """Convert configuration to semicolon-separated string for CSV header"""
        config_dict = {
            'software': 'mouseReMoCo',
            'version': '2.0.0-python',
            'screenWidth': self.config.screen_width,
            'screenHeight': self.config.screen_height,
            'centerX': self.config.center_x,
            'centerY': self.config.center_y,
            'isWithLSL': str(self.config.is_with_lsl).lower(),
        }
        
        # Add task-specific configuration
        if self.config.task_string == 'circular':
            config_dict.update({
                'externalRadius': self.config.external_radius,
                'internalRadius': self.config.internal_radius,
                'internalLimit': self.config.internal_limit,
                'externalLimit': self.config.external_limit,
                'borderRadius': self.config.border_radius,
                'indexOfDifficulty': round(self.config.index_of_difficulty, 2),
            })
        
        return ';'.join([f"{k} {v}" for k, v in config_dict.items()])
    
    def write_data(self, timestamp_ms: int, x: float, y: float, is_inside: bool, 
                   pressure: float = 0.0, tilt_x: float = 0.0, tilt_y: float = 0.0):
        """Write position data to CSV with subpixel precision (2 decimal places)"""
        try:
            self.data_writer.writerow([
                timestamp_ms,
                round(x, 2),  # Subpixel precision: 2 decimal places
                round(y, 2),  # Subpixel precision: 2 decimal places
                1 if is_inside else 0,
                round(pressure, 3),
                round(tilt_x, 2),
                round(tilt_y, 2)
            ])
            self.data_file.flush()
        except Exception as e:
            print(f"ERROR writing data to {self.data_filename}: {e}")
    
    def write_marker(self, marker_text: str):
        """Write event marker to CSV"""
        try:
            current_time = datetime.now()
            timestamp_ms = int(current_time.timestamp() * 1000)
            timestamp_str = current_time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            
            self.marker_writer.writerow([
                timestamp_str,
                timestamp_ms,
                marker_text
            ])
            self.marker_file.flush()
        except Exception as e:
            print(f"ERROR writing marker to {self.marker_filename}: {e}")
    
    def close(self):
        """Close CSV files"""
        try:
            if self.data_file:
                self.data_file.close()
                print(f"✓ Closed {self.data_filename}")
            if self.marker_file:
                self.marker_file.close()
                print(f"✓ Closed {self.marker_filename}")
        except Exception as e:
            print(f"ERROR closing CSV files: {e}")


class LSLBackend(OutputBackend):
    """LSL (Lab Streaming Layer) output backend with subpixel support"""
    
    def __init__(self, config):
        self.config = config
        self.data_outlet = None
        self.marker_outlet = None
        self.numeric_marker_outlet = None
        
        try:
            import lsl
            self.lsl = lsl
            self._init_lsl()
            print("✓ LSL Backend: Initialized successfully")
        except ImportError:
            print("⚠ LSL library not available - LSL backend disabled")
            self.lsl = None
    
    def _init_lsl(self):
        """Initialize LSL streams for data and markers"""
        if not self.lsl:
            return
        
        # Data stream (float32 supports subpixel precision)
        data_info = self.lsl.StreamInfo(
            name='MouseData',
            type='MoCap',
            channel_count=3,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_float32,
            source_id='mouseReMoCo'
        )
        
        # Add channel descriptions
        chns = data_info.desc().append_child('channels')
        labels = ['mouseX', 'mouseY', 'mouseInTarget']
        types = ['PositionX', 'PositionY', 'flag']
        units = ['pixels', 'pixels', 'boolean']
        
        for label, type_, unit in zip(labels, types, units):
            chns.append_child('channel') \
                .append_child_value('label', label) \
                .append_child_value('type', type_) \
                .append_child_value('unit', unit)
        
        self.data_outlet = self.lsl.StreamOutlet(data_info)
        
        # Marker stream
        marker_info = self.lsl.StreamInfo(
            name='MouseMarkers',
            type='Markers',
            channel_count=1,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_string,
            source_id='mouseReMoCo_markers'
        )
        self.marker_outlet = self.lsl.StreamOutlet(marker_info)
        
        # Numeric marker stream (for sync)
        numeric_info = self.lsl.StreamInfo(
            name='MouseMarkersNumeric',
            type='Markers',
            channel_count=1,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_int32,
            source_id='mouseReMoCo_markers_numeric'
        )
        self.numeric_marker_outlet = self.lsl.StreamOutlet(numeric_info)
    
    def write_data(self, timestamp_ms: int, x: float, y: float, is_inside: bool, 
                   pressure: float = 0.0, tilt_x: float = 0.0, tilt_y: float = 0.0):
        """Push position data to LSL with full subpixel precision"""
        if not self.lsl or not self.data_outlet:
            return
        
        try:
            # LSL float32 preserves subpixel precision
            sample = [float(x), float(y), float(1 if is_inside else 0)]
            self.data_outlet.push_sample(sample)
        except Exception as e:
            print(f"ERROR pushing data to LSL: {e}")
    
    def write_marker(self, marker_text: str):
        """Push event marker to LSL"""
        if not self.lsl or not self.marker_outlet:
            return
        
        try:
            sample = [marker_text]
            self.marker_outlet.push_sample(sample)
        except Exception as e:
            print(f"ERROR pushing marker to LSL: {e}")
    
    def close(self):
        """Close LSL outlets"""
        if not self.lsl:
            return
        
        try:
            if self.data_outlet:
                self.data_outlet = None
            if self.marker_outlet:
                self.marker_outlet = None
            if self.numeric_marker_outlet:
                self.numeric_marker_outlet = None
            print("✓ Closed LSL Backend")
        except Exception as e:
            print(f"ERROR closing LSL: {e}")


class OutputTablet:
    """Unified output manager with pluggable backends (CSV, LSL) - Subpixel support"""
    
    def __init__(self, config, enable_csv: bool = True, enable_lsl: bool = False):
        """
        Initialize OutputTablet with desired backends.
        
        Args:
            config: Configuration object with screen and task parameters
            enable_csv: Enable CSV file output with subpixel precision (default: True)
            enable_lsl: Enable LSL streaming output (default: False)
        """
        self.config = config
        self.backends = []
        
        if enable_csv:
            self.backends.append(CSVBackend(config))
        
        if enable_lsl:
            lsl_backend = LSLBackend(config)
            if lsl_backend.lsl:  # Only add if LSL initialized successfully
                self.backends.append(lsl_backend)
        
        if not self.backends:
            raise ValueError("At least one backend must be enabled (CSV or LSL)")
    
    def write_data(self, timestamp_ms: int, x: float, y: float, is_inside: bool, 
                   pressure: float = 0.0, tilt_x: float = 0.0, tilt_y: float = 0.0):
        """Write position data with subpixel precision to all active backends"""
        for backend in self.backends:
            backend.write_data(timestamp_ms, x, y, is_inside, pressure, tilt_x, tilt_y)
    
    def write_marker(self, marker_text: str):
        """Write event marker to all active backends"""
        for backend in self.backends:
            backend.write_marker(marker_text)
    
    def close(self):
        """Close all backends gracefully"""
        for backend in self.backends:
            backend.close()
    
    def __del__(self):
        """Ensure backends are closed when object is garbage collected"""
        self.close()

In [ ]:
# MainWindow widget - Main application window with tablet input and comet trail

import sys

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QColor, QPainter, QPen, QTabletEvent
from PyQt6.QtWidgets import QApplication, QWidget


class MainWindow(QWidget):
    # Map keyboard keys to trail modes from Trail class
    # Dynamically built to stay in sync if Trail.VALID_MODES changes
    TRAIL_MODES = {str(i + 1): mode for i, mode in enumerate(Trail.VALID_MODES)}

    def __init__(
        self,
        screen_info,
        circle_config,
        usable_width: int,
        usable_height: int,
        actual_drawable_width: int = None,
        actual_drawable_height: int = None,
        config: "Configuration" = None,
        window_setup: "WindowSetup" = None,
        output_data: "OutputTablet" = None, 
    ):
        super().__init__()
        self.screen_info = screen_info
        self.config = config
        self.window_setup = window_setup
        self.circular_target = CircularTargetWidget(config=circle_config)
        self.usable_width = usable_width
        self.usable_height = usable_height
        # Use actual drawable dimensions if provided, otherwise use usable dimensions
        self.drawable_width = actual_drawable_width or usable_width
        self.drawable_height = actual_drawable_height or usable_height
        self.center_x = usable_width // 2
        self.center_y = usable_height // 2

        # Comet trail management: independent trails for mouse and tablet input
        self.trail_mouse = Trail(config, is_mouse=True)   # Fixed 5px width
        self.trail_tablet = Trail(config, is_mouse=False)  # Pressure-scaled
        self.active_trail = self.trail_mouse              # Track which trail to display
        self.last_mouse_x = None
        self.last_mouse_y = None
        
        # Accept OutputData passed from WindowSetup (created after config correction)
        self.output_data = output_data  # ← CHANGE THIS
        if self.output_data:
            self.trail_mouse.output_data = self.output_data
            self.trail_tablet.output_data = self.output_data

        # Fullscreen mode: 0=windowed, 1=fullscreen, 2=borderless fullscreen
        self.fullscreen_mode = 0

        # Tablet detection on first pen proximity
        self.tablet_detected_flag = False

        # Enable mouse tracking to receive mouseMoveEvent even when no button is pressed
        self.setMouseTracking(True)
        self.setFocus()

        print(
            f"\n{'='*60}\nTrail Mode Controls:\n"
            f"Press 1: Fixed Distance\n"
            f"Press 2: Pressure Scaled\n"
            f"Press 3: Speed Adaptive\n"
            f"Press 4: Time Based\n"
            f"Press 5: Point Count\n"
            f"Press 6: Combined (Pressure + Speed)\n"
            f"Press 7: Path Length\n"
            f"Press F: Toggle Fullscreen (Windowed ↔ Borderless)\n"
            f"Press C: Print Configuration\n"
            f"Press Q: Quit\n"
            f"{'='*60}\n"
        )

    def paintEvent(self, event):
        painter = QPainter(self)
        # Use background color from config, or default to black
        if self.config and self.config.background_color:
            bg_color = QColor(*self.config.background_color)
        else:
            bg_color = Qt.GlobalColor.black
        painter.fillRect(self.rect(), bg_color)

        # Draw limits rectangles for debugging
        self._draw_limits_retangles(painter)

        # Draw the circular target
        self.circular_target.draw(painter, self.center_x, self.center_y)

        # Draw only the active trail (automatically switched on input)
        self.active_trail.draw(painter, self.center_x, self.center_y)

        # Draw mode indicator on screen
        self._draw_mode_indicator(painter)

        self._draw_limits_retangles(painter)

    def _draw_limits_retangles(self, painter: QPainter):
        """Draw non-filled rectangles showing drawable screen limits"""
        # Draw green-yellow rectangles showing drawable screen limits

        original_brush = painter.brush()
        painter.setBrush(Qt.BrushStyle.NoBrush)

        shift = 0  # small shift to see the border more clearly (-1,suppresses the green rect)
        painter.setPen(QPen(Qt.GlobalColor.green, 1))
        painter.drawRect(
            shift,
            shift + 1,  # drawRect needs this correction (test on OSx)
            self.drawable_width - 2 * shift,
            self.drawable_height - 2 * shift - 1,
        )
        shift += 5
        painter.setPen(QPen(Qt.GlobalColor.yellow, 1))
        painter.drawRect(
            shift,
            shift + 1,
            self.drawable_width - 2 * (shift),
            self.drawable_height - 2 * (shift) - 1,
        )

        painter.setBrush(original_brush)

    def _draw_mode_indicator(self, painter: QPainter):
        """Draw current trail mode in corner"""
        mode_text = f"Mode: {self.config.trail_mode.upper()}"
        painter.setPen(QPen(Qt.GlobalColor.white))
        painter.drawText(10, 20, mode_text)

    def _toggle_fullscreen(self):
        """Toggle between windowed and borderless fullscreen with dimension recomputation"""
        if self.fullscreen_mode == 0:
            # Switch to borderless fullscreen (no system UI)
            self.setWindowFlags(Qt.WindowType.FramelessWindowHint)
            self.setGeometry(self.screen().geometry())
            self.showFullScreen()
            self.fullscreen_mode = 1
            print("\nToggled to BORDERLESS FULLSCREEN (no system UI access)")
        else:
            # Return to windowed
            self.setWindowFlags(Qt.WindowType.Widget)
            self.showNormal()
            self.fullscreen_mode = 0
            print("\nToggled to WINDOWED mode")

        # Process events to apply window state changes
        QApplication.instance().processEvents()

        # Recalculate drawable dimensions using WindowSetup's method
        if self.window_setup:
            self.window_setup.measure_and_correct_dimensions()

        self.setFocus()

    def _quit_application(self):
        """Gracefully quit application, handling fullscreen state"""
        # Disable all input to suppress user interaction during shutdown
        self.setEnabled(False)

        # If in fullscreen, toggle to windowed first, wait 1 sec, then close
        # (fullscreen close is buggy on macOS, so bypass by going windowed first)
        if self.fullscreen_mode == 1:
            self._toggle_fullscreen()
            # Delay close by 1 second to let window state settle
            from PyQt6.QtCore import QTimer

            QTimer.singleShot(1000, self.close)
        else:
            # Already windowed, close immediately
            self.close()

    def closeEvent(self, event):
        """Handle window close - exit fullscreen before closing"""
        # If in fullscreen, explicitly return to windowed BEFORE closing
        if self.fullscreen_mode == 1:
            self.setWindowFlags(Qt.WindowType.Widget)
            self.showNormal()
            self.fullscreen_mode = 0
            # CRITICAL: Let windowing system process state changes before accepting close
            QApplication.instance().processEvents()

        # Close output data files before exiting
        if self.output_data:
            self.output_data.close()
        event.accept()

    def showEvent(self, event):
        """Initialize cursor when widget is shown"""
        super().showEvent(event)
        # Set initial cursor to "out" state
        self.setCursor(CursorFactory.create_out_cursor(self.config))

    def _update_cursor_for_position(self, x: float, y: float):
        """Update cursor color based on distance from circle center"""
        dx = self.center_x - x
        dy = self.center_y - y
        distance = (dx * dx + dy * dy) ** 0.5

        is_inside = self.config.internal_limit < distance < self.config.external_limit

        if is_inside:
            self.setCursor(CursorFactory.create_record_cursor(self.config))
        else:
            self.setCursor(CursorFactory.create_out_cursor(self.config))

    def tabletEvent(self, event: QTabletEvent):
        """Route tablet input to tablet trail and activate it"""
        # Detect tablet on first pen proximity (auto-proximity detection)
        if not self.tablet_detected_flag and event.pressure() > 0:
            self.tablet_detected_flag = True
            self.output_data.write_marker('TabletDetected')  # ADD THIS LINE
            print("\n" + "="*60)
            print("✓ Tablet detected and active!")
            print("="*60)
        
        # Clear mouse trail when switching from mouse to tablet
        if self.active_trail is self.trail_mouse:
            self.trail_mouse.clear()
        
        self.trail_tablet.current_pressure = event.pressure()
        x = event.position().x() # float - subpixel precision
        y = event.position().y()
        self.trail_tablet.add_point(x, y)
        self.active_trail = self.trail_tablet  # Activate tablet trail
        self._update_cursor_for_position(x, y)
        self.update()  # Trigger repaint
        event.accept()

    def mouseMoveEvent(self, event):
        """Route mouse input to mouse trail and activate it"""
        # Clear tablet trail when switching from tablet to mouse
        if self.active_trail is self.trail_tablet:
            self.trail_tablet.clear()
        
        x = event.position().x() # float - subpixel precision
        y = event.position().y() # float - subpixel precision

        # Add point to mouse trail (handles speed, distance tracking, and pruning)
        self.trail_mouse.add_point(x, y)
        self.active_trail = self.trail_mouse  # Activate mouse trail

        self._update_cursor_for_position(x, y)

        self.last_mouse_x = x
        self.last_mouse_y = y

        # Trigger repaint to draw updated trail
        self.update()

    def mousePressEvent(self, event):
        """Print mouse click position"""
        print(
            f"Mouse click at: X={event.position().x():.1f}, Y={event.position().y():.1f}"
        )
        sys.stdout.flush()

    def keyPressEvent(self, event):
        """Handle keyboard input for trail mode switching and app control"""
        key = event.text()

        # Trail mode switching (1-7)
        if key in self.TRAIL_MODES:
            new_mode = self.TRAIL_MODES[key]
            # Update both trails when mode changes
            self.trail_mouse.set_mode(new_mode)
            self.trail_tablet.set_mode(new_mode)
            print(f"\n{'='*60}")
            print(f"Switched to trail mode: {new_mode.upper()}")
            print(f"{'='*60}\n")
            self.update()

        # Quit application
        elif key.lower() == "q":
            self._quit_application()

        # Print configuration
        elif key.lower() == "c":
            print("\n" + "=" * 60)
            print(self.config.to_string())
            print("=" * 60 + "\n")

        # Toggle fullscreen
        elif key.lower() == "f":
            self._toggle_fullscreen()

        else:
            super().keyPressEvent(event)

In [ ]:
# Run Application — Execute the mouseReMoCo Wacom Tablet Test

# Create or retrieve the Qt application singleton
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

# Initialize window orchestrator (creates Configuration internally)
# Handles: screen detection, dimension calculation, configuration lifecycle, window creation
window_setup = WindowSetup(
    app=app,
    tablet_test_class=MainWindow,
)

# Execute complete initialization pipeline
# Detects screens → calculates circle radii → creates window →
# measures frame insets → corrects drawable dimensions → displays window
main_window = window_setup.create_and_display()

# Update configuration MUST be after main_window knows drawable area
window_setup.update_configuration(
    trail_length=2
    * 3.14159
    * window_setup.config.internal_radius  # ~ 1 lap
    # index_of_difficulty=70.0,  # sets circle perimeter accordingly
)

# Launch the Qt event loop
# Blocks until user closes the window; handles all input and rendering
exit_code = app.exec()